# Log Data Processing
This notebook processes an OCEL dataset stored in a SQLite
database (`CargoPickup_IoT.sqlite`). It merges events, E2O/O2O relations, truck data, pickup-plan
data, and cargo data into a single flat table, segments the pickup path and processes the GPS data to generate three csv files suitable for downstream process-mining analysis.

# 1. Import & Helper Utilities

In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np

DATA_DIR = 'data'
DB_PATH = os.path.join(DATA_DIR, 'CargoPickup_IoT.sqlite')

In [2]:
def process_simple_event(conn, table_name, e2o_df):
    """Event → merge e2o → [event_id, time, object_id]"""
    # Load event table
    event_df = pd.read_sql_query(f"SELECT ocel_id, ocel_time FROM {table_name}", conn)
    event_df = event_df.drop_duplicates()
    
    # Merge with e2o
    merged = pd.merge(event_df, e2o_df, left_on='ocel_id', right_on='ocel_event_id', how='inner')
    
    # Select columns and return
    res = merged[['ocel_event_id', 'ocel_time', 'ocel_object_id']].copy()
    res = res.drop_duplicates()
    return res

def process_iot_event(conn, table_name, e2o_df, e2iot_df, extra_cols=None):
    """Event → merge e2o + e2iot → [event_id, time, object_id, IoT_object_id]"""
    if extra_cols is None:
        extra_cols = []
    
    cols = ['ocel_id', 'ocel_time'] + extra_cols
    col_str = ', '.join([f'"{c}"' for c in cols])
    event_df = pd.read_sql_query(f"SELECT {col_str} FROM {table_name}", conn)
    event_df = event_df.drop_duplicates()
    
    # Merge with e2o
    merged_e2o = pd.merge(event_df, e2o_df, left_on='ocel_id', right_on='ocel_event_id', how='inner')
    
    # Merge with e2iot
    merged_all = pd.merge(merged_e2o, e2iot_df, on='ocel_event_id', how='inner')
    merged_all = merged_all.drop_duplicates()
    
    final_cols = ['ocel_event_id', 'ocel_time', 'ocel_object_id', 'ocel_IoT_object_id'] + extra_cols
    res = merged_all[final_cols].copy()
    return res

def process_o2o_event(conn, table_name, e2o_df, o2o_df, is_iot=False, e2iot_df=None):
    """Event → merge e2o (and optional e2iot) + o2o → reshape source/target"""
    event_df = pd.read_sql_query(f"SELECT ocel_id, ocel_time FROM {table_name}", conn)
    event_df = event_df.drop_duplicates()
    
    # Merge with e2o
    merged_e2o = pd.merge(event_df, e2o_df, left_on='ocel_id', right_on='ocel_event_id', how='inner')
    
    if is_iot:
        # Merge with e2iot
        merged_base = pd.merge(merged_e2o, e2iot_df, on='ocel_event_id', how='inner')
    else:
        merged_base = merged_e2o
        
    merged_base = merged_base.drop_duplicates()
    
    # Determine which column to join with o2o
    join_side = 'source'
    if table_name == 'event_DetermineContinuanceofPickup':
        join_side = 'target'
        
    left_keys = ['ocel_object_id', 'ocel_time']
    right_keys = [f'ocel_{join_side}_id', 'ocel_time']
    
    merged_all = pd.merge(merged_base, o2o_df, left_on=left_keys, right_on=right_keys, how='inner')
    merged_all = merged_all.drop_duplicates()
    
    # Reshape: source_df + target_df
    base_cols = ['ocel_event_id', 'ocel_time']
    if is_iot:
        base_cols.append('ocel_IoT_object_id')
        
    source_df = merged_all[base_cols + ['ocel_source_id']].rename(columns={'ocel_source_id': 'ocel_object_id'})
    target_df = merged_all[base_cols + ['ocel_target_id']].rename(columns={'ocel_target_id': 'ocel_object_id'})
    
    reshaped = pd.concat([source_df, target_df])
    reshaped = reshaped.sort_values(by='ocel_time').reset_index(drop=True)
    reshaped = reshaped.drop_duplicates()
    
    if is_iot:
        reshaped = reshaped[['ocel_event_id', 'ocel_time', 'ocel_object_id', 'ocel_IoT_object_id']]
        
    return reshaped

## 2. Connection

In [3]:
conn = sqlite3.connect(DB_PATH)

## 3. Base Tables

In [4]:
e2o_df = pd.read_sql_query("SELECT ocel_event_id, ocel_object_id, ocel_qualifier FROM event_object", conn)
o2o_df = pd.read_sql_query("SELECT ocel_source_id, ocel_target_id, ocel_qualifier, ocel_time FROM object_object", conn)
e2iot_df = pd.read_sql_query("SELECT ocel_event_id, ocel_IoT_object_id, ocel_qualifier FROM event_IoTobject", conn)
o2iot_df = pd.read_sql_query("SELECT ocel_IoTobject_id, ocel_object_id, ocel_qualifier, ocel_time FROM object_IoTobject", conn)

## 4. Process Events

### Simple Events

In [5]:
lodge_df = process_simple_event(conn, 'event_LodgePickupPlan', e2o_df)
assess_df = process_simple_event(conn, 'event_AssessPickupPlan', e2o_df)
approve_df = process_simple_event(conn, 'event_ApprovePickupPlan', e2o_df)
register_arr_df = process_simple_event(conn, 'event_RegisterTruckArrival', e2o_df)
fail_weigh_df = process_simple_event(conn, 'event_FailtoWeigh', e2o_df)
load_df = process_simple_event(conn, 'event_LoadTruck', e2o_df)
eval_exit_df = process_simple_event(conn, 'event_EvaluateTruckExit', e2o_df)
tally_df = process_simple_event(conn, 'event_InputTallySheet', e2o_df)
issue_ticket_df = process_simple_event(conn, 'event_IssueWeighingTicket', e2o_df)
fail_load_df = process_simple_event(conn, 'event_FailtoLoad', e2o_df)
detain_df = process_simple_event(conn, 'event_DetainAtPort', e2o_df)

### IoT Events

In [6]:
entry_df = process_iot_event(conn, 'event_Enterport', e2o_df, e2iot_df, extra_cols=['rain condition'])
entry_df = entry_df.rename(columns={'rain condition': 'rain_condition'})
check_df = process_iot_event(conn, 'event_CheckEmptyTruckWeightAbnormality', e2o_df, e2iot_df)

### O2O Events

In [7]:
assign_df = process_o2o_event(conn, 'event_AssignTruck', e2o_df, o2o_df)
arrive_silo_df = process_o2o_event(conn, 'event_ArriveatSilo', e2o_df, o2o_df)
exit_df = process_o2o_event(conn, 'event_ExitPort', e2o_df, o2o_df)
determine_df = process_o2o_event(conn, 'event_DetermineContinuanceofPickup', e2o_df, o2o_df, is_iot=True, e2iot_df=e2iot_df)

### Special

In [8]:
# WeighEmptyTruck
weigh_empty_event = pd.read_sql_query("SELECT ocel_id, ocel_time FROM event_WeighEmptyTruck", conn)
weigh_empty_event = weigh_empty_event.drop_duplicates()
ws_empty = o2iot_df[(o2iot_df['ocel_IoTobject_id'].str.startswith('WS')) & (o2iot_df['ocel_qualifier'].str.contains('empty', case=False))]
empty_merged = pd.merge(ws_empty, weigh_empty_event, on='ocel_time', how='inner')
empty_merged['tr_from_id'] = empty_merged['ocel_id'].str.extract(r'(tr\d+)')
empty_merged['tr_from_object'] = empty_merged['ocel_object_id']
empty_merged = empty_merged[empty_merged['tr_from_id'] == empty_merged['tr_from_object']]
emptyw_df = empty_merged[['ocel_id', 'ocel_time', 'ocel_object_id', 'ocel_IoTobject_id']].rename(columns={'ocel_id': 'ocel_event_id', 'ocel_IoTobject_id': 'ocel_IoT_object_id'})

# WeighLoadedTruck
weigh_loaded_event = pd.read_sql_query("SELECT ocel_id, ocel_time FROM event_WeighLoadedTruck", conn)
weigh_loaded_event['ocel_object_id'] = weigh_loaded_event['ocel_id'].str.extract(r'(tr\d+)')
ws_loaded = o2iot_df[(o2iot_df['ocel_IoTobject_id'].str.startswith('WS')) & (o2iot_df['ocel_qualifier'].str.contains('loaded', case=False))]
loaded_merged = pd.merge(ws_loaded, weigh_loaded_event, on=['ocel_time', 'ocel_object_id'], how='inner')
loaded_merged = loaded_merged.drop_duplicates()
loadedw_df = loaded_merged[['ocel_id', 'ocel_time', 'ocel_object_id', 'ocel_IoTobject_id']].rename(columns={'ocel_id': 'ocel_event_id', 'ocel_IoTobject_id': 'ocel_IoT_object_id'})

## 5. Concatenate All Event Tables

In [9]:
df_3col = [lodge_df, assess_df, approve_df, assign_df, register_arr_df, fail_weigh_df, arrive_silo_df, load_df, eval_exit_df, tally_df, issue_ticket_df, fail_load_df, detain_df, exit_df]
df_4col = [entry_df, emptyw_df, check_df, determine_df, loadedw_df]

merged_df = pd.concat(df_3col + df_4col, ignore_index=True)
events_sorted = merged_df.sort_values(by='ocel_time').reset_index(drop=True)
events_sorted.head()

,ocel_event_id,ocel_time,ocel_object_id,ocel_IoT_object_id,rain_condition
0,lodge_Pcp497,2024-04-29 07:25:58,Pcp497,NaN,NaN
1,lodge_Pcp497,2024-04-29 07:25:58,Cr14,NaN,NaN
2,lodge_Pcp465,2024-04-29 07:28:42,Pcp465,NaN,NaN
3,lodge_Pcp465,2024-04-29 07:28:42,Cr18,NaN,NaN
4,lodge_Pcp156,2024-04-29 07:40:59,Pcp156,NaN,NaN


## 6. Activity Mapping

In [10]:
event_names_df = pd.read_sql_query("SELECT ocel_id AS ocel_event_id, ocel_type FROM event", conn)
event_names_df = event_names_df.drop_duplicates(subset='ocel_event_id')

final_events_df = pd.merge(events_sorted, event_names_df, on='ocel_event_id', how='left')
final_events_df = final_events_df.rename(columns={'ocel_event_id': 'event_id', 'ocel_type': 'activity', 'ocel_time': 'timestamp'})
final_events_df.head()

,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity
0,lodge_Pcp497,2024-04-29 07:25:58,Pcp497,NaN,NaN,Lodge Pickup Plan
1,lodge_Pcp497,2024-04-29 07:25:58,Cr14,NaN,NaN,Lodge Pickup Plan
2,lodge_Pcp465,2024-04-29 07:28:42,Pcp465,NaN,NaN,Lodge Pickup Plan
3,lodge_Pcp465,2024-04-29 07:28:42,Cr18,NaN,NaN,Lodge Pickup Plan
4,lodge_Pcp156,2024-04-29 07:40:59,Pcp156,NaN,NaN,Lodge Pickup Plan


## 7. Load Object Tables

In [11]:
truck_cols = ['ocel_id AS ocel_object_id', 'ocel_time AS timestamp', 'ocel_changed_field', '"RFID No"', '"Pickup Plan ID"', '"Cargo ID"', 'LPT', 'Axles', '"Scheduled Pickup Weight"', '"Truck Status"', '"Truck Location Validity"', '"Truck Weight"', 'Truck_Weight_Status']
truck_df = pd.read_sql_query(f"SELECT {', '.join(truck_cols)} FROM object_Truck", conn)

pp_cols = ['ocel_id AS ocel_object_id', 'ocel_time AS timestamp', 'ocel_changed_field', '"Cargo ID"', '"Num of trucks"', '"Total pickup weight"']
pp_df = pd.read_sql_query(f"SELECT {', '.join(pp_cols)} FROM object_Pickupplan", conn)

cargo_cols = ['ocel_id AS ocel_object_id', 'ocel_time AS timestamp', 'ocel_changed_field', '"Cargo Type"', '"Cargo stock weight(withholding)"', '"Cargo stock weight(real)"', '"Silo ID"']
cargo_df = pd.read_sql_query(f"SELECT {', '.join(cargo_cols)} FROM object_Cargo", conn)

## 8. Merge Object Tables

In [12]:
trucks_merged_df = pd.merge(final_events_df, truck_df, on=['ocel_object_id', 'timestamp'], how='outer')
pp_merged_truck = pd.merge(trucks_merged_df, pp_df, on=['ocel_object_id', 'timestamp'], how='outer')
pp_merged_truck.head()

,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field_x,RFID No,Pickup Plan ID,Cargo ID_x,...,Axles,Scheduled Pickup Weight,Truck Status,Truck Location Validity,Truck Weight,Truck_Weight_Status,ocel_changed_field_y,Cargo ID_y,Num of trucks,Total pickup weight
0,lodge_Pcp483,2024-04-29 18:29:34,Cr1,NaN,NaN,Lodge Pickup Plan,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lodge_Pcp130,2024-04-30 07:09:09,Cr1,NaN,NaN,Lodge Pickup Plan,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lodge_Pcp35,2024-04-30 23:53:42,Cr1,NaN,NaN,Lodge Pickup Plan,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lodge_Pcp436,2024-05-01 09:54:33,Cr1,NaN,NaN,Lodge Pickup Plan,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,lodge_Pcp220,2024-05-02 06:24:28,Cr1,NaN,NaN,Lodge Pickup Plan,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Handle Lodge Pickup Plan special logic

In [13]:
lodge_mask = pp_merged_truck['activity'] == 'Lodge Pickup Plan'
lodge_merged_filtered = pp_merged_truck[lodge_mask].copy()

lodge_merged_filtered['Cargo ID_x'] = lodge_merged_filtered['Cargo ID_x'].fillna(lodge_merged_filtered['Cargo ID_y'])
pcps_mask = lodge_merged_filtered['Pickup Plan ID'].isna() & lodge_merged_filtered['ocel_object_id'].str.startswith('Pcp')
lodge_merged_filtered.loc[pcps_mask, 'Pickup Plan ID'] = lodge_merged_filtered.loc[pcps_mask, 'ocel_object_id']
lodge_merged_filtered = lodge_merged_filtered[~lodge_merged_filtered['ocel_object_id'].str.startswith('Cr', na=False)]
lodge_merged_filtered = lodge_merged_filtered.drop(columns=['Cargo ID_y', 'ocel_changed_field_y'])
lodge_merged_filtered = lodge_merged_filtered.rename(columns={'Cargo ID_x': 'Cargo ID', 'ocel_changed_field_x': 'ocel_changed_field'})

non_lodge_merged_filtered = pp_merged_truck[~lodge_mask].copy()
non_lodge_merged_filtered = non_lodge_merged_filtered.drop(columns=['Cargo ID_y', 'ocel_changed_field_y'])
non_lodge_merged_filtered = non_lodge_merged_filtered.rename(columns={'Cargo ID_x': 'Cargo ID', 'ocel_changed_field_x': 'ocel_changed_field'})

lodge_merged_trucks = pd.concat([lodge_merged_filtered, non_lodge_merged_filtered]).reset_index(drop=True)

lodge_merged_trucks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42762 entries, 0 to 42761
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   event_id                 14262 non-null  object 
 1   timestamp                42762 non-null  object 
 2   ocel_object_id           42762 non-null  object 
 3   ocel_IoT_object_id       6781 non-null   object 
 4   rain_condition           489 non-null    object 
 5   activity                 14262 non-null  object 
 6   ocel_changed_field       32373 non-null  object 
 7   RFID No                  50 non-null     object 
 8   Pickup Plan ID           758 non-null    object 
 9   Cargo ID                 758 non-null    object 
 10  LPT                      50 non-null     object 
 11  Axles                    50 non-null     float64
 12  Scheduled Pickup Weight  539 non-null    float64
 13  Truck Status             1028 non-null   object 
 14  Truck Location Validit

In [14]:
cargo_merged_rest = pd.merge(lodge_merged_trucks, cargo_df, on=['ocel_object_id', 'timestamp'], how='outer')
cargo_merged_rest['ocel_changed_field_x'] = cargo_merged_rest['ocel_changed_field_x'].fillna(cargo_merged_rest['ocel_changed_field_y'])
cargo_merged_rest = cargo_merged_rest.drop(columns=['ocel_changed_field_y'])
cargo_merged_rest = cargo_merged_rest.rename(columns={'ocel_changed_field_x': 'ocel_changed_field'})
cargo_merged_rest.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43540 entries, 0 to 43539
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   event_id                         14262 non-null  object 
 1   timestamp                        43540 non-null  object 
 2   ocel_object_id                   43540 non-null  object 
 3   ocel_IoT_object_id               6781 non-null   object 
 4   rain_condition                   489 non-null    object 
 5   activity                         14262 non-null  object 
 6   ocel_changed_field               33131 non-null  object 
 7   RFID No                          50 non-null     object 
 8   Pickup Plan ID                   758 non-null    object 
 9   Cargo ID                         758 non-null    object 
 10  LPT                              50 non-null     object 
 11  Axles                            50 non-null     float64
 12  Scheduled Pickup W

### Backfill missing values for Lodge Pickup Plan from related Cargo records

In [15]:
lodge_ts = cargo_merged_rest[cargo_merged_rest['activity'] == 'Lodge Pickup Plan']['timestamp'].unique()
    
update_mask = (cargo_merged_rest['activity'] == 'Lodge Pickup Plan') | ((cargo_merged_rest['event_id'].isna()) & (cargo_merged_rest['timestamp'].isin(lodge_ts)) & cargo_merged_rest['ocel_object_id'].str.startswith('Cr'))
df_update_part = cargo_merged_rest[update_mask].copy()
df_rest = cargo_merged_rest[~update_mask].copy()

lodge_mask_update = df_update_part['activity'] == 'Lodge Pickup Plan'
lodge_idxs = df_update_part[lodge_mask_update].index
filler_df = df_update_part[~lodge_mask_update]

for idx in lodge_idxs:
    ts = df_update_part.at[idx, 'timestamp']
    matching_rows = filler_df[filler_df['timestamp'] == ts]
    for _, mrow in matching_rows.iterrows():
        for col in cargo_merged_rest.columns:
            val = df_update_part.at[idx, col]
            if pd.isna(val) or val in ['', 'None', 'nan', 'NaN']:
                if not pd.isna(mrow[col]) and mrow[col] not in ['', 'None', 'nan', 'NaN']:
                    df_update_part.at[idx, col] = mrow[col]
                    
df_final = pd.concat([df_update_part[lodge_mask_update], df_rest], ignore_index=True)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43271 entries, 0 to 43270
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   event_id                         14262 non-null  object 
 1   timestamp                        43271 non-null  object 
 2   ocel_object_id                   43271 non-null  object 
 3   ocel_IoT_object_id               6781 non-null   object 
 4   rain_condition                   489 non-null    object 
 5   activity                         14262 non-null  object 
 6   ocel_changed_field               33131 non-null  object 
 7   RFID No                          50 non-null     object 
 8   Pickup Plan ID                   758 non-null    object 
 9   Cargo ID                         758 non-null    object 
 10  LPT                              50 non-null     object 
 11  Axles                            50 non-null     float64
 12  Scheduled Pickup W

## 7. Post-Processing Activity Labels

In [16]:
df_final.loc[(df_final['ocel_changed_field'] == '') & df_final['LPT'].notna(), 'activity'] = 'Review the Truck Information'
df_final.loc[df_final['ocel_changed_field'] == 'Truck Location Validity', 'activity'] = 'Track Truck Location'
df_final.loc[(df_final['ocel_changed_field'] == '') & df_final['Cargo Type'].notna(), 'activity'] = 'Review the Cargo Information'
df_final.loc[(df_final['ocel_changed_field'] == 'Cargo stock weight(withholding)') & (df_final['activity'].isna()), 'activity'] = 'Update Cargo Stock Withhold'
df_final.loc[(df_final['ocel_changed_field'] == 'Cargo stock weight(real)') & (df_final['activity'].isna()), 'activity'] = 'Update Cargo Real Stock'

### Merge grouped rows helper

In [17]:
def merge_group_rows_with_field(group):
    merged_row = group.iloc[0].copy()
    for col in group.columns:
        if col not in ['event_id', 'activity', 'timestamp', 'ocel_object_id', 'ocel_IoT_object_id', 'ocel_changed_field']:
            non_null_values = group[col].dropna().unique()
            if len(non_null_values) > 0:
                merged_row[col] = non_null_values[0]
    for _, row in group.iterrows():
        field = row['ocel_changed_field']
        if field in group.columns and pd.notna(row[field]):
            merged_row[field] = row[field]
    return merged_row

### Assign Truck Merge

In [18]:
assign_mask = df_final['activity'] == 'Assign Truck'
assign_trucks_df = df_final[assign_mask]
assign_trucks_df = assign_trucks_df[assign_trucks_df['ocel_object_id'].str.startswith('tr', na=False)]
other_activities_df = df_final[~assign_mask]

assigned = assign_trucks_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_group_rows_with_field).reset_index(drop=True)
all_xes = pd.concat([assigned, other_activities_df]).sort_values(by='timestamp').reset_index(drop=True)
all_xes

/var/folders/_8/t9lzz28d1nx60l3whmwkckv00000gn/T/ipykernel_4601/2487592625.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  assigned = assign_trucks_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_group_rows_with_field).reset_index(drop=True)


,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field,RFID No,Pickup Plan ID,Cargo ID,...,Truck Status,Truck Location Validity,Truck Weight,Truck_Weight_Status,Num of trucks,Total pickup weight,Cargo Type,Cargo stock weight(withholding),Cargo stock weight(real),Silo ID
0,NaN,2024-03-04 00:00:00,tr2,NaN,NaN,NaN,NaN,rfidSQKJGH,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2024-03-04 00:00:00,tr38,NaN,NaN,NaN,NaN,rfidVIASXP,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2024-03-04 00:00:00,tr8,NaN,NaN,NaN,NaN,rfidCEZEBQ,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2024-03-04 00:00:00,tr25,NaN,NaN,NaN,NaN,rfidMFTSWW,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2024-03-04 00:00:00,tr27,NaN,NaN,NaN,NaN,rfidWOMLHU,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41310,NaN,2024-05-31 08:50:31,Cr19,NaN,NaN,Update Cargo Real Stock,Cargo stock weight(real),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,None
41311,eval_tr8exit_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Evaluate the Truck Exit,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41312,input_tally_tr8_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Input the Tally Sheet,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41313,exit_tr8_Pcp235,2024-05-31 08:51:12,tr8,NaN,NaN,Exit the port,Truck Status,None,None,None,...,Available,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN


### Fail to Weigh Merge

In [19]:
fail_mask = all_xes['activity'] == 'Fail to Weigh'
fail_df = all_xes[fail_mask]
rest_df = all_xes[~fail_mask]

if not fail_df.empty:
    fail = fail_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_group_rows_with_field).reset_index(drop=True)
    all_df1 = pd.concat([fail, rest_df]).sort_values(by='timestamp').reset_index(drop=True)
else:
    all_df1 = all_xes.sort_values(by='timestamp').reset_index(drop=True)

all_df1

/var/folders/_8/t9lzz28d1nx60l3whmwkckv00000gn/T/ipykernel_4601/1892936802.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fail = fail_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_group_rows_with_field).reset_index(drop=True)


,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field,RFID No,Pickup Plan ID,Cargo ID,...,Truck Status,Truck Location Validity,Truck Weight,Truck_Weight_Status,Num of trucks,Total pickup weight,Cargo Type,Cargo stock weight(withholding),Cargo stock weight(real),Silo ID
0,NaN,2024-03-04 00:00:00,Cr15,NaN,NaN,NaN,None,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Wheat,132312.3,132312.3,Silo16
1,NaN,2024-03-04 00:00:00,tr5,NaN,NaN,NaN,NaN,rfidSVWFVZ,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2024-03-04 00:00:00,tr22,NaN,NaN,NaN,NaN,rfidENJOLW,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2024-03-04 00:00:00,tr10,NaN,NaN,NaN,NaN,rfidOOVDPE,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2024-03-04 00:00:00,Cr6,NaN,NaN,NaN,None,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Rice,137171.7,137171.7,Silo12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41307,NaN,2024-05-31 08:50:31,Cr19,NaN,NaN,Update Cargo Real Stock,Cargo stock weight(real),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,None
41308,eval_tr8exit_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Evaluate the Truck Exit,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41309,input_tally_tr8_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Input the Tally Sheet,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41310,exit_tr8_Pcp235,2024-05-31 08:51:12,tr8,NaN,NaN,Exit the port,Truck Status,None,None,None,...,Available,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN


### Arrive at Silo Merge

In [20]:
def merge_rows_silo(group):
    if len(group) == 2:
        row1, row2 = group.iloc[0].copy(), group.iloc[1].copy()
        if str(row1['ocel_object_id']).startswith('Silo'):
            row2['Silo ID'] = row1['ocel_object_id']
            return row2
        elif str(row2['ocel_object_id']).startswith('Silo'):
            row1['Silo ID'] = row2['ocel_object_id']
            return row1
        else:
            return row1  
    else:
        return group.iloc[0]

arrive_mask = all_df1['activity'] == 'Arrive at the Silo'
arriveSilo_df = all_df1[arrive_mask]
non_arriveSilo_df = all_df1[~arrive_mask]

merged_silo = arriveSilo_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_rows_silo).reset_index(drop=True)
grouped_arrival = pd.concat([merged_silo, non_arriveSilo_df], ignore_index=True)
grouped_arrival

/var/folders/_8/t9lzz28d1nx60l3whmwkckv00000gn/T/ipykernel_4601/875595572.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_silo = arriveSilo_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_rows_silo).reset_index(drop=True)


,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field,RFID No,Pickup Plan ID,Cargo ID,...,Truck Status,Truck Location Validity,Truck Weight,Truck_Weight_Status,Num of trucks,Total pickup weight,Cargo Type,Cargo stock weight(withholding),Cargo stock weight(real),Silo ID
0,arrive_Silo10_tr11_Pcp408,2024-05-14 11:04:32,tr11,NaN,NaN,Arrive at the Silo,Truck_Weight_Status,None,None,None,...,None,None,NaN,normal,NaN,NaN,NaN,NaN,NaN,Silo10
1,arrive_Silo10_tr17_Pcp343,2024-05-20 16:47:49,tr17,NaN,NaN,Arrive at the Silo,Truck_Weight_Status,None,None,None,...,None,None,NaN,normal,NaN,NaN,NaN,NaN,NaN,Silo10
2,arrive_Silo10_tr18_Pcp214,2024-05-20 14:42:37,tr18,NaN,NaN,Arrive at the Silo,Truck_Weight_Status,None,None,None,...,None,None,NaN,normal,NaN,NaN,NaN,NaN,NaN,Silo10
3,arrive_Silo10_tr19_Pcp131,2024-05-14 11:23:32,tr19,NaN,NaN,Arrive at the Silo,Truck_Weight_Status,None,None,None,...,None,None,NaN,normal,NaN,NaN,NaN,NaN,NaN,Silo10
4,arrive_Silo10_tr19_Pcp186,2024-05-18 13:34:42,tr19,NaN,NaN,Arrive at the Silo,Truck_Weight_Status,None,None,None,...,None,None,NaN,normal,NaN,NaN,NaN,NaN,NaN,Silo10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40821,NaN,2024-05-31 08:50:31,Cr19,NaN,NaN,Update Cargo Real Stock,Cargo stock weight(real),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,None
40822,eval_tr8exit_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Evaluate the Truck Exit,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40823,input_tally_tr8_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Input the Tally Sheet,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40824,exit_tr8_Pcp235,2024-05-31 08:51:12,tr8,NaN,NaN,Exit the port,Truck Status,None,None,None,...,Available,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN


### Determine Continuance Silo propagation

In [21]:
determine_mask = grouped_arrival['activity'] == 'Determine the Continuance of the Pickup'
determine_df = grouped_arrival[determine_mask].copy()
non_determine_df = grouped_arrival[~determine_mask]

silo_rows = determine_df[determine_df['ocel_object_id'].str.startswith('Silo', na=False)]
for _, silo_row in silo_rows.iterrows():
    timestamp = silo_row['timestamp']
    silo_id = silo_row['ocel_object_id']
    mask = (determine_df['timestamp'] == timestamp) & determine_df['ocel_object_id'].str.startswith('tr', na=False)
    determine_df.loc[mask, 'Silo ID'] = silo_id
    
determine_df = determine_df[~determine_df['ocel_object_id'].str.startswith('Silo', na=False)].reset_index(drop=True)
grouped_determine = pd.concat([determine_df, non_determine_df], ignore_index=True)
grouped_determine

,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field,RFID No,Pickup Plan ID,Cargo ID,...,Truck Status,Truck Location Validity,Truck Weight,Truck_Weight_Status,Num of trucks,Total pickup weight,Cargo Type,Cargo stock weight(withholding),Cargo stock weight(real),Silo ID
0,determine_continu_tr38_Pcp35_Silo3,2024-05-03 00:16:02,tr38,GrainTemp9,NaN,Determine the Continuance of the Pickup,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Silo3
1,determine_continu_tr38_Pcp35_Silo3,2024-05-03 00:16:02,tr38,Temp3,NaN,Determine the Continuance of the Pickup,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Silo3
2,determine_continu_tr38_Pcp35_Silo3,2024-05-03 00:16:02,tr38,GrainTemp8,NaN,Determine the Continuance of the Pickup,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Silo3
3,determine_continu_tr38_Pcp35_Silo3,2024-05-03 00:16:02,tr38,Humd3,NaN,Determine the Continuance of the Pickup,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Silo3
4,determine_continu_tr38_Pcp35_Silo3,2024-05-03 00:16:02,tr38,GrainTemp7,NaN,Determine the Continuance of the Pickup,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Silo3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38391,NaN,2024-05-31 08:50:31,Cr19,NaN,NaN,Update Cargo Real Stock,Cargo stock weight(real),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,None
38392,eval_tr8exit_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Evaluate the Truck Exit,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38393,input_tally_tr8_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Input the Tally Sheet,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38394,exit_tr8_Pcp235,2024-05-31 08:51:12,tr8,NaN,NaN,Exit the port,Truck Status,None,None,None,...,Available,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN


### Exit Port Merge

In [22]:
def merge_rows_exit(group):
    if len(group) == 2:
        row1, row2 = group.iloc[0].copy(), group.iloc[1].copy()
        if str(row1['ocel_object_id']).startswith('Pcp'):
            row2['Pickup Plan ID'] = row1['ocel_object_id']
            return row2
        elif str(row2['ocel_object_id']).startswith('Pcp'):
            row1['Pickup Plan ID'] = row2['ocel_object_id']
            return row1
        else:
            return row1  
    else:
        return group.iloc[0]

exit_mask = grouped_determine['activity'] == 'Exit the port'
exit_df = grouped_determine[exit_mask]
non_exit_df = grouped_determine[~exit_mask]

merged_exit = exit_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_rows_exit).reset_index(drop=True)
grouped_exit = pd.concat([merged_exit, non_exit_df], ignore_index=True).sort_values('timestamp')
grouped_exit

/var/folders/_8/t9lzz28d1nx60l3whmwkckv00000gn/T/ipykernel_4601/3539517206.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_exit = exit_df.groupby(['event_id', 'timestamp'], group_keys=False).apply(merge_rows_exit).reset_index(drop=True)


,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field,RFID No,Pickup Plan ID,Cargo ID,...,Truck Status,Truck Location Validity,Truck Weight,Truck_Weight_Status,Num of trucks,Total pickup weight,Cargo Type,Cargo stock weight(withholding),Cargo stock weight(real),Silo ID
3414,NaN,2024-03-04 00:00:00,tr23,NaN,NaN,NaN,NaN,rfidKGKHYU,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
3395,NaN,2024-03-04 00:00:00,tr3,NaN,NaN,NaN,NaN,rfidDRCUXI,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
3396,NaN,2024-03-04 00:00:00,tr19,NaN,NaN,NaN,NaN,rfidRCKRLL,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
3397,NaN,2024-03-04 00:00:00,tr16,NaN,NaN,NaN,NaN,rfidIICVAH,None,None,...,Available,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN
3398,NaN,2024-03-04 00:00:00,Cr20,NaN,NaN,NaN,None,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Rice,123405.7,123405.7,Silo9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37939,NaN,2024-05-31 08:50:31,Cr19,NaN,NaN,Update Cargo Real Stock,Cargo stock weight(real),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,None
37938,issue_ticket_tr8_Pcp235,2024-05-31 08:50:31,tr8,NaN,NaN,Issue the Weighing Ticket,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37941,input_tally_tr8_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Input the Tally Sheet,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37940,eval_tr8exit_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Evaluate the Truck Exit,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Propagate Cargo and Pickup Plan IDs based on ocel_object_id

In [23]:
crs = grouped_exit['ocel_object_id'].str.startswith('Cr', na=False)
grouped_exit.loc[crs, 'Cargo ID'] = grouped_exit.loc[crs, 'ocel_object_id']

pcps = grouped_exit['ocel_object_id'].str.startswith('Pcp', na=False)
grouped_exit.loc[pcps, 'Pickup Plan ID'] = grouped_exit.loc[pcps, 'ocel_object_id']

grouped_exit['Truck ID'] = grouped_exit['ocel_object_id'].where(grouped_exit['ocel_object_id'].str.startswith('tr', na=False), None)
grouped_exit

,event_id,timestamp,ocel_object_id,ocel_IoT_object_id,rain_condition,activity,ocel_changed_field,RFID No,Pickup Plan ID,Cargo ID,...,Truck Location Validity,Truck Weight,Truck_Weight_Status,Num of trucks,Total pickup weight,Cargo Type,Cargo stock weight(withholding),Cargo stock weight(real),Silo ID,Truck ID
3414,NaN,2024-03-04 00:00:00,tr23,NaN,NaN,NaN,NaN,rfidKGKHYU,None,None,...,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,tr23
3395,NaN,2024-03-04 00:00:00,tr3,NaN,NaN,NaN,NaN,rfidDRCUXI,None,None,...,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,tr3
3396,NaN,2024-03-04 00:00:00,tr19,NaN,NaN,NaN,NaN,rfidRCKRLL,None,None,...,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,tr19
3397,NaN,2024-03-04 00:00:00,tr16,NaN,NaN,NaN,NaN,rfidIICVAH,None,None,...,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,tr16
3398,NaN,2024-03-04 00:00:00,Cr20,NaN,NaN,NaN,None,NaN,NaN,Cr20,...,NaN,NaN,NaN,NaN,NaN,Rice,123405.7,123405.7,Silo9,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37939,NaN,2024-05-31 08:50:31,Cr19,NaN,NaN,Update Cargo Real Stock,Cargo stock weight(real),NaN,NaN,Cr19,...,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,None,None
37938,issue_ticket_tr8_Pcp235,2024-05-31 08:50:31,tr8,NaN,NaN,Issue the Weighing Ticket,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tr8
37941,input_tally_tr8_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Input the Tally Sheet,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tr8
37940,eval_tr8exit_Pcp235,2024-05-31 08:50:32,tr8,NaN,NaN,Evaluate the Truck Exit,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tr8


## 8. Pickup Tracking

In [24]:
trucks = grouped_exit[grouped_exit['ocel_object_id'].str.startswith('tr', na=False)]
non_trucks = grouped_exit[~grouped_exit['ocel_object_id'].str.startswith('tr', na=False)].copy()

truck_ordered = trucks.sort_values(by=['ocel_object_id', 'timestamp']).reset_index(drop=True)
truck_ordered['pickup'] = None

pickup_id = 0
assigning_pickup = False
current_truck = None

for idx, row in truck_ordered.iterrows():
    activity = row['activity']
    truck_id = row['ocel_object_id']
    
    if activity == "Assign Truck":
        pickup_id += 1
        assigning_pickup = True
        current_truck = truck_id
        truck_ordered.at[idx, 'pickup'] = pickup_id
    elif assigning_pickup and truck_id == current_truck:
        truck_ordered.at[idx, 'pickup'] = pickup_id
        if activity in ["Exit the port", "Fail to Weigh", "Fail to Load"]:
            assigning_pickup = False
            
pp_cr_ids = truck_ordered.dropna(subset=['pickup']).drop_duplicates(subset=['pickup'])[['pickup', 'Pickup Plan ID', 'Cargo ID']]
alltrucks_df = pd.merge(truck_ordered, pp_cr_ids, on='pickup', how='outer', suffixes=('', '_filled'))
alltrucks_df['Cargo ID'] = alltrucks_df['Cargo ID'].fillna(alltrucks_df['Cargo ID_filled'])
alltrucks_df = alltrucks_df.drop(columns=['Cargo ID_filled'])

non_trucks['Pickup Plan ID_filled'] = None
groupPP = non_trucks['Pickup Plan ID_filled'].isna() & non_trucks['Pickup Plan ID'].notnull()
non_trucks.loc[groupPP, 'Pickup Plan ID_filled'] = non_trucks.loc[groupPP, 'Pickup Plan ID']

all_final_df = pd.concat([alltrucks_df, non_trucks], ignore_index=True)
all_final_df = all_final_df.drop(columns=['ocel_changed_field', 'Pickup Plan ID'])
all_final_df = all_final_df.rename(columns={'Pickup Plan ID_filled': 'Pickup Plan ID'})

all_final_df['Cargo ID'] = all_final_df['Cargo ID'].replace(['', ' '], pd.NA)
all_final_df['Cargo ID'] = all_final_df.groupby('pickup')['Cargo ID'].transform(lambda x: x.ffill().bfill())

crs1 = all_final_df['ocel_object_id'].str.startswith('Cr', na=False)
all_final_df.loc[crs1, 'Cargo ID'] = all_final_df.loc[crs1, 'ocel_object_id']
all_final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37942 entries, 0 to 37941
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   event_id                         8933 non-null   object 
 1   timestamp                        37942 non-null  object 
 2   ocel_object_id                   37942 non-null  object 
 3   ocel_IoT_object_id               4351 non-null   object 
 4   rain_condition                   489 non-null    object 
 5   activity                         37872 non-null  object 
 6   RFID No                          50 non-null     object 
 7   Cargo ID                         37085 non-null  object 
 8   LPT                              50 non-null     object 
 9   Axles                            50 non-null     float64
 10  Scheduled Pickup Weight          539 non-null    float64
 11  Truck Status                     1028 non-null   object 
 12  Truck Location Val

### Generate All_pickups and Success_pickups

In [25]:
def add_duration_columns(df, group_col='pickup'):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    df['Enter_RegisterArrival'] = None
    df['Register_EmptyWeighing'] = None
    df['EmptyWeighing_ArrivalSilo'] = None
    df['ArrivalSilo_Determine'] = None
    df['Determine_Stack'] = None
    df['Stack_Loaded'] = None
    df['LoadedWeighing_Eval'] = None
    df['Eval_Exit'] = None
    
    for group_id, group in df.groupby(group_col):
        def get_duration(start_act, end_act):
            start = group.loc[group['activity'] == start_act, 'timestamp']
            end = group.loc[group['activity'] == end_act, 'timestamp']
            if not start.empty and not end.empty:
                return (end.iloc[0] - start.iloc[0]).total_seconds() / 60
            return None

        df.loc[group.index, 'Enter_RegisterArrival'] = get_duration("Enter the port", "Register the truck arrival")
        df.loc[group.index, 'Register_EmptyWeighing'] = get_duration("Register the truck arrival", "Weigh the Empty Truck")
        df.loc[group.index, 'EmptyWeighing_ArrivalSilo'] = get_duration("Weigh the Empty Truck","Arrive at the Silo")
        df.loc[group.index, 'ArrivalSilo_Determine'] = get_duration("Arrive at the Silo", "Determine the Continuance of the Pickup")
        df.loc[group.index, 'Determine_Stack'] = get_duration ("Determine the Continuance of the Pickup","Load Truck")
        df.loc[group.index, 'Stack_Loaded'] = get_duration("Load Truck", "Weigh the Loaded Truck")
        df.loc[group.index, 'LoadedWeighing_Eval'] = get_duration("Weigh the Loaded Truck", "Evaluate the Truck Exit")
        df.loc[group.index, 'Eval_Exit'] = get_duration("Evaluate the Truck Exit", "Exit the port")

    return df

def add_track_location_counts(df, group_col='pickup'):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df['Enter_to_Register_TTL_count'] = 0
    df['Register_to_WeighEmpty_TTL_count'] = 0
    df['WeighEmpty_to_ArriveSilo_TTL_count'] = 0
    df['Stack_Loaded_TTL_count'] = 0
    df['WeighLoaded_to_Exit_TTL_count'] = 0

    for pickup_id, group in df.groupby(group_col):
        group = group.sort_values('timestamp')
    
        def count_ttl_between_activities(group, start_act, end_act):
            start_idx = group.index[group['activity'] == start_act]
            end_idx = group.index[group['activity'] == end_act]

            if not start_idx.empty and not end_idx.empty:
                start_pos = start_idx[0]
                end_pos = end_idx[0]
        
                if start_pos < end_pos:
                    between = group.loc[start_pos:end_pos]
                    return (between['activity'] == 'Track Truck Location').sum()
            return 0

        df.loc[group.index, 'Enter_to_Register_TTL_count'] = count_ttl_between_activities(group, 'Enter the port', 'Register the truck arrival')
        df.loc[group.index, 'Register_to_WeighEmpty_TTL_count'] = count_ttl_between_activities(group, 'Register the truck arrival', 'Weigh the Empty Truck')
        df.loc[group.index, 'WeighEmpty_to_ArriveSilo_TTL_count'] = count_ttl_between_activities(group, 'Weigh the Empty Truck', 'Arrive at the Silo')
        df.loc[group.index, 'Stack_Loaded_TTL_count'] = count_ttl_between_activities(group, 'Load Truck','Weigh the Loaded Truck')
        df.loc[group.index, 'WeighLoaded_to_Exit_TTL_count'] = count_ttl_between_activities(group, 'Weigh the Loaded Truck', 'Evaluate the Truck Exit')

    return df

In [26]:
# Compute duration & TTL counts
df_durations = add_duration_columns(all_final_df)
df_durations_gps = add_track_location_counts(df_durations)

# All_pickups
all_pickups = df_durations_gps[(df_durations_gps['pickup'].notnull()) & (df_durations_gps['activity'] != 'Assign Truck')].copy()

# All_success_pickups
failed_ids = all_pickups[all_pickups['activity'].isin(['Fail to Load', 'Fail to Weigh'])]['pickup'].unique()
success_pickups = all_pickups[~all_pickups['pickup'].isin(failed_ids)].copy()

### GPS Data

In [27]:
# Load Data
rows = []
with open(os.path.join(DATA_DIR, 'TruckGPSRecord.csv'), "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        if len(parts) == 6:
            rows.append(parts)
df_GPS_raw = pd.DataFrame(rows, columns=["GPS_ID", "RFID tag", "Pickup Plan", "Timestamp", "Latitude", "Longitude"])
df_GPS_raw["Timestamp"] = pd.to_datetime(df_GPS_raw["Timestamp"])
df_GPS_raw["Latitude"] = pd.to_numeric(df_GPS_raw["Latitude"], errors='coerce')
df_GPS_raw["Longitude"] = pd.to_numeric(df_GPS_raw["Longitude"], errors='coerce')
df_GPS_raw["Date"] = df_GPS_raw["Timestamp"].dt.date

# Path Stage Classification
path_regions = {
    "entry_to_register": {"lat_min": 21.9860, "lat_max": 21.9940, "lon_min": 100.4890, "lon_max": 100.4930},
    "register_to_weigh": {"lat_min": 21.9945, "lat_max": 22.0005, "lon_min": 100.4950, "lon_max": 100.52},
    "weigh_to_yard":     {"lat_min": 22.0080, "lat_max": 22.0150, "lon_min": 100.5250, "lon_max": 100.5450},
    "yard_to_weigh":     {"lat_min": 21.9880, "lat_max": 22.0075, "lon_min": 100.5555, "lon_max": 100.5650},
    "weigh_to_exit":     {"lat_min": 21.98, "lat_max": 21.9850, "lon_min": 100.5655, "lon_max": 100.5700},
}

def assign_path_stage(row):
    for stage, bounds in path_regions.items():
        if (bounds["lat_min"] <= row["Latitude"] <= bounds["lat_max"] and
            bounds["lon_min"] <= row["Longitude"] <= bounds["lon_max"]):
            return stage
    return "off_path"

df_GPS_raw["Path_Stage"] = df_GPS_raw.apply(assign_path_stage, axis=1)

# Prepare GPS data for mergence in the later analysis
valid_rfid_df = truck_df[truck_df['RFID No'].notna() & (truck_df['RFID No'] != '')]
truck_to_rfid = valid_rfid_df[['ocel_object_id', 'RFID No']].drop_duplicates(subset='ocel_object_id', keep='first').set_index('ocel_object_id')['RFID No'].to_dict()

df_GPS_raw['RFID tag'] = df_GPS_raw['RFID tag'].astype(str).str.strip()
rfid_to_truck = {v.strip(): k for k, v in truck_to_rfid.items() if isinstance(v, str)}
df_GPS_raw['Truck ID'] = df_GPS_raw['RFID tag'].map(rfid_to_truck)
df_GPS_raw = df_GPS_raw.rename(columns={'Pickup Plan':'Pickup Plan ID'})
df_GPS_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38592 entries, 0 to 38591
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   GPS_ID          38592 non-null  object        
 1   RFID tag        38592 non-null  object        
 2   Pickup Plan ID  38592 non-null  object        
 3   Timestamp       38592 non-null  datetime64[ns]
 4   Latitude        38592 non-null  float64       
 5   Longitude       38592 non-null  float64       
 6   Date            38592 non-null  object        
 7   Path_Stage      38592 non-null  object        
 8   Truck ID        38592 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 2.7+ MB


### Save Output

In [29]:
all_pickups.to_csv(os.path.join(DATA_DIR, 'all_pickups.csv'), index=False)
print(f"Successfully processed {len(all_pickups)} rows. Saved to all_pickups.csv")

success_pickups.to_csv(os.path.join(DATA_DIR, 'success_pickups.csv'), index=False)
print(f"Successfully processed {len(success_pickups)} rows. Saved to success_pickups.csv")

df_GPS_raw.to_csv(os.path.join(DATA_DIR, 'gps_processed.csv'), index=False)
print(f"Successfully processed {len(df_GPS_raw)} rows. Saved to gps_merged.csv")

Successfully processed 36087 rows. Saved to all_pickups.csv
Successfully processed 34503 rows. Saved to success_pickups.csv
Successfully processed 38592 rows. Saved to gps_merged.csv
